In [ ]:
import fastf1
import plotly.graph_objects as go
import yaml

with open("C:/Users/boncl/Repos/simply-lovely/config/teams_drivers_info.yaml", "r") as file:
    config = yaml.safe_load(file)
fastf1.Cache.configure(cache_dir='C:/Users/boncl/Repos/simply-lovely/fastf1_cache')

compound_colours = {
    "SOFT": "#E10600",
    "MEDIUM": "#FFD12E",
    "HARD": "#FAF9F6",
    "INTERMEDIATE": "#43B02A",
    "WET": "#0067AD"
}

In [12]:
silverstone_laps = fastf1.get_session(2026, 'Bahrain', 'R')
silverstone_laps.load()
laps = silverstone_laps.laps
laps

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '3'
core        WARNING 	Fixed incorrect tyre stint information for driver '44'
core        WARNING 	Fixed incorrect tyre stint information for driver '16'
core        WARNING 	Fixed incorrect tyre stint information for driver '6'
core        WARNING 	Fixed incorrect tyre stint information for driver '81'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WA

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 02:28:23.340000,NOR,1,0 days 00:02:41.802000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:48.640000,...,True,McLaren,0 days 02:26:00.700000,2026-10-04 08:33:00.115,1,4.0,False,,False,False
1,0 days 02:32:15.212000,NOR,1,NaT,2.0,1.0,NaT,NaT,0 days 00:00:38.158000,0 days 00:00:52.129000,...,True,McLaren,0 days 02:28:23.340000,2026-10-04 08:35:22.755,12,10.0,False,,False,False
2,0 days 02:34:35.261000,NOR,1,0 days 00:02:20.049000,3.0,1.0,NaT,NaT,0 days 00:00:36.515000,0 days 00:00:48.885000,...,True,McLaren,0 days 02:32:15.212000,2026-10-04 08:39:14.627,1,16.0,False,,False,True
3,0 days 02:36:42.033000,NOR,1,0 days 00:02:06.772000,4.0,1.0,NaT,NaT,0 days 00:00:31.456000,0 days 00:00:44.259000,...,True,McLaren,0 days 02:34:35.261000,2026-10-04 08:41:34.676,1,15.0,True,TRACK LIMITS AT TURN 12 LAP 4,False,True
4,0 days 02:38:44.009000,NOR,1,0 days 00:02:01.976000,5.0,1.0,NaT,NaT,0 days 00:00:30.442000,0 days 00:00:42.599000,...,True,McLaren,0 days 02:36:42.033000,2026-10-04 08:43:41.448,1,15.0,False,,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1140,0 days 04:06:28.518000,BEA,87,NaT,51.0,4.0,NaT,NaT,0 days 00:00:38.083000,0 days 00:00:52.601000,...,False,Haas F1 Team,0 days 04:03:57.988000,2026-10-04 10:10:57.403,41,12.0,False,,False,False
1141,0 days 04:08:11.895000,BEA,87,0 days 00:01:43.377000,52.0,4.0,NaT,NaT,0 days 00:00:26.286000,0 days 00:00:35.707000,...,False,Haas F1 Team,0 days 04:06:28.518000,2026-10-04 10:13:27.933,1,13.0,False,,False,True
1142,0 days 04:09:53.691000,BEA,87,0 days 00:01:41.796000,53.0,4.0,NaT,NaT,0 days 00:00:25.757000,0 days 00:00:34.841000,...,False,Haas F1 Team,0 days 04:08:11.895000,2026-10-04 10:15:11.310,1,13.0,False,,False,True
1143,0 days 04:11:35.983000,BEA,87,0 days 00:01:42.292000,54.0,4.0,NaT,NaT,0 days 00:00:26.022000,0 days 00:00:34.785000,...,False,Haas F1 Team,0 days 04:09:53.691000,2026-10-04 10:16:53.106,1,13.0,False,,False,True


In [53]:
fig = go.Figure()

for driver in laps["Driver"].unique():
    driver_laps = laps[laps["Driver"] == driver]
    driver_team = laps.loc[laps['Driver']== driver, 'Team'].unique()[0]
    fig.add_scatter(
        x=driver_laps["LapNumber"],
        y=driver_laps["Position"],
        mode="lines",
        name=driver,
        line=dict(
            color=f"rgba({config['teams'][driver_team]['drivers'][driver]['colour']})"
        ))

fig.update_yaxes(autorange="reversed")
fig.show()

In [54]:
fig2 = go.Figure()

driver_order = (
    laps.groupby("Driver")["Position"]
    .last()
    .sort_values()
    .index
    .tolist()
)

for driver in laps["Driver"].unique():
    driver_laps = laps[laps["Driver"] == driver]

    for _, lap in driver_laps.iterrows():
        fig2.add_trace(
            go.Bar(
                x = [1],
                y = [driver],
                base = lap['LapNumber'] -1,
                orientation='h',
                marker_color = compound_colours[lap['Compound']],
                marker_line_width=0,
                showlegend= False
            )
        )

fig2.update_layout(
    barmode="stack",
    xaxis_title="Lap",
    yaxis_title="Driver",
    title = "Tyre strategies over Silverstone GP",
    template="plotly_white",
    bargap=0.2,
    height = 600
)

fig2.update_yaxes(
    categoryorder="array",
    categoryarray=driver_order,
    autorange="reversed"
)
fig2.update_xaxes(showgrid = True, gridwidth = 1, gridcolor = 'lightgrey')

fig2.show()

In [ ]:
#TODO: fix fastest driver lap. mayebt he line could show the average fastest lap, and the fastest lap itself should just be info next to it?

fig3 = go.Figure()
laps_violin = laps.pick_quicklaps()
fastest_lap_overall = laps_violin['LapTime'].dt.total_seconds().min()
fastest_driver = laps_violin.loc[laps_violin['LapTime'] == fastest_lap_overall, 'Driver']
print(fastest_driver)
average_lap_time_overall = laps_violin['LapTime'].dt.total_seconds().mean()

for driver in laps_violin['Driver'].unique():
    driver_laps = laps_violin[laps_violin['Driver'] == driver]
    driver_mean = driver_laps['LapTime'].dt.total_seconds().mean()
    driver_team = driver_laps['Team'].iat[0]
    driver_colour = config['teams'][driver_team]['drivers'][driver]['colour']

    fig3.add_trace(go.Violin(
        y=driver_laps['LapTime'].dt.total_seconds(),
        name= driver,
        showlegend= False,
        line=dict(color=f"rgba({driver_colour})")
    ))
    fig3.add_trace(go.Scatter(
        y = [driver_mean],
        x = [driver],
        mode ="markers",
        marker= dict(symbol="diamond", size = 5, color=f"rgba({driver_colour})"
                ),
        showlegend = False)
    )

fig3.add_hline(y = fastest_lap_overall, line_dash="dash", line_color = "red", annotation_text=f"Fastest Lap by {fastest_driver}: {fastest_lap_overall} s")
fig3.update_yaxes(title="Lap Time in s")
fig3.update_xaxes(title= "Driver")
fig3.update_layout(title = "Spread of Lap Time per driver over race")
fig3.show()

Series([], Name: Driver, dtype: object)


In [8]:
silverstone_laps = fastf1.get_session(2026, 'Silverstone', 'R')
silverstone_laps.load()

positions_info = silverstone_laps.pos_data
print(positions_info.keys())
positions_info

core           INFO 	Loading data for British Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['16', '63', '44', '1', '6', '30', '41', '5', '43', '10', '81', '87', '31', '11', '12', '77', '55', '14', '18', '3', '23', '27']


dict_keys(['16', '63', '44', '1', '6', '30', '41', '5', '43', '10', '81', '87', '31', '11', '12', '77', '55', '14', '18', '3', '23', '27'])


{'16':                          Date   Status       X      Y       Z Source  \
 0     2026-07-05 13:08:29.001  OnTrack     0.0    0.0     0.0    pos   
 1     2026-07-05 13:08:29.241  OnTrack     0.0    0.0     0.0    pos   
 2     2026-07-05 13:08:29.481  OnTrack     0.0    0.0     0.0    pos   
 3     2026-07-05 13:08:29.701  OnTrack     0.0    0.0     0.0    pos   
 4     2026-07-05 13:08:29.861  OnTrack     0.0    0.0     0.0    pos   
 ...                       ...      ...     ...    ...     ...    ...   
 34520 2026-07-05 15:39:28.687  OnTrack -1622.0  721.0  1986.0    pos   
 34521 2026-07-05 15:39:29.207  OnTrack -1622.0  721.0  1986.0    pos   
 34522 2026-07-05 15:39:29.427  OnTrack -1622.0  721.0  1986.0    pos   
 34523 2026-07-05 15:39:29.647  OnTrack -1622.0  721.0  1986.0    pos   
 34524 2026-07-05 15:39:29.907  OnTrack -1622.0  721.0  1986.0    pos   
 
                         Time            SessionTime  
 0     0 days 00:00:14.508000 0 days 00:00:14.508000  
 1    

In [9]:
import pandas as pd

all_positions = []

for driver, df in silverstone_laps.pos_data.items():

    temp = df.copy()

    temp["Driver"] = driver

    all_positions.append(temp)

all_positions = pd.concat(all_positions, ignore_index=True)

In [ ]:
laps = silverstone_laps.laps
laps = laps.sort_values(by=['Driver', 'LapNumber'])
laps['Previous_Position'] = laps.groupby('Driver')['Position'].shift(1)
laps['Position_Change'] = (
    laps['Previous_Position'] - laps['Position']
)
gained_places = laps[laps['Position_Change'] > 0]
lost_places = laps[laps['Position_Change'] < 0]
overtakes = []

for _, gain in gained_places.iterrows():

    lap = gain["LapNumber"]

    old_pos = gain["Previous_Position"]
    new_pos = gain["Position"]
    candidate = lost_places[
    (lost_places["LapNumber"] == lap)
    & (lost_places["Previous_Position"] == new_pos)
    & (lost_places["Position"] == old_pos)
]
    if candidate.empty:
        continue
    passed_driver = candidate.iloc[0]["Driver"]
    passing_driver = gain["Driver"]
    overtakes.append({
    "LapNumber": lap,
    "PassingDriver": passing_driver,
    "PassedDriver": passed_driver,
    "OldPosition": old_pos,
    "NewPosition": new_pos
})

overtakes = pd.DataFrame(overtakes)

for _, overtake in overtakes.iterrows():
    lap = overtake["LapNumber"]
    overtaking = overtake["PassingDriver"]
    overtaken = overtake["PassedDriver"]
    overtaking_telemetry = (laps[(laps['LapNumber']==lap) & (laps['Driver'] == overtaking)].iloc[0]).get_telemetry()
    overtaken_telemetry = (laps[(laps['LapNumber']==lap) & (laps['Driver'] == overtaken)].iloc[0]).get_telemetry()

[{'LapNumber': 19.0, 'PassingDriver': 'ALO', 'PassedDriver': 'STR'},
 {'LapNumber': 22.0, 'PassingDriver': 'ALO', 'PassedDriver': 'STR'},
 {'LapNumber': 11.0, 'PassingDriver': 'ANT', 'PassedDriver': 'HAM'},
 {'LapNumber': 26.0, 'PassingDriver': 'ANT', 'PassedDriver': 'LEC'},
 {'LapNumber': 6.0, 'PassingDriver': 'BEA', 'PassedDriver': 'BOT'},
 {'LapNumber': 23.0, 'PassingDriver': 'BEA', 'PassedDriver': 'COL'},
 {'LapNumber': 31.0, 'PassingDriver': 'BEA', 'PassedDriver': 'PER'},
 {'LapNumber': 34.0, 'PassingDriver': 'BEA', 'PassedDriver': 'OCO'},
 {'LapNumber': 5.0, 'PassingDriver': 'BOR', 'PassedDriver': 'SAI'},
 {'LapNumber': 5.0, 'PassingDriver': 'COL', 'PassedDriver': 'OCO'},
 {'LapNumber': 9.0, 'PassingDriver': 'COL', 'PassedDriver': 'SAI'},
 {'LapNumber': 49.0, 'PassingDriver': 'COL', 'PassedDriver': 'GAS'},
 {'LapNumber': 3.0, 'PassingDriver': 'GAS', 'PassedDriver': 'OCO'},
 {'LapNumber': 6.0, 'PassingDriver': 'GAS', 'PassedDriver': 'SAI'},
 {'LapNumber': 48.0, 'PassingDriver': 'G

In [ ]:
from scipy.spatial import cKDTree
import numpy as np
ref_lap = silverstone_laps.laps.pick_fastest()

ref_tel = ref_lap.get_telemetry()

ref_xy = np.column_stack([
    ref_tel["X"].to_numpy(),
    ref_tel["Y"].to_numpy()
])

tree = cKDTree(ref_xy)

ref_distance = ref_tel["Distance"].to_numpy()

In [ ]:
def add_track_distance(telemetry, tree, ref_distance):

    xy = np.column_stack([
        telemetry["X"].to_numpy(),
        telemetry["Y"].to_numpy()
    ])

    _, idx = tree.query(xy)

    telemetry = telemetry.copy()

    telemetry["TrackDistance"] = ref_distance[idx]

    return telemetry

telemetry = add_track_distance(
    telemetry,
    tree,
    ref_distance
)

In [ ]:
import fastf1
import pandas as pd
import numpy as np
c

# --- Load session ---
silverstone_laps = fastf1.get_session(2026, 'Silverstone', 'R')
silverstone_laps.load()

# --- Build lap-based position change table ---
laps = silverstone_laps.laps.copy()
laps = laps.sort_values(by=['Driver', 'LapNumber'])

laps['Previous_Position'] = laps.groupby('Driver')['Position'].shift(1)
laps['Position_Change'] = laps['Previous_Position'] - laps['Position']

gained_places = laps[laps['Position_Change'] > 0]
lost_places   = laps[laps['Position_Change'] < 0]

# --- Detect overtakes ---
overtakes = []

for _, gain in gained_places.iterrows():
    lap = gain["LapNumber"]
    old_pos = gain["Previous_Position"]
    new_pos = gain["Position"]

    candidate = lost_places[
        (lost_places["LapNumber"] == lap)
        & (lost_places["Previous_Position"] == new_pos)
        & (lost_places["Position"] == old_pos)
    ]

    if candidate.empty:
        continue

    overtakes.append({
        "LapNumber": lap,
        "PassingDriver": gain["Driver"],
        "PassedDriver": candidate.iloc[0]["Driver"],
        "OldPosition": old_pos,
        "NewPosition": new_pos
    })

overtakes = pd.DataFrame(overtakes)

# --- Build KDTree for track distance mapping ---
ref_lap = silverstone_laps.laps.pick_fastest()
ref_tel = ref_lap.get_telemetry()
ref_tel = add_track_distance(ref_tel, tree, ref_distance)


ref_xy = np.column_stack([ref_tel["X"].to_numpy(),
                          ref_tel["Y"].to_numpy()])

tree = cKDTree(ref_xy)
ref_distance = ref_tel["Distance"].to_numpy()

def add_track_distance(telemetry, tree, ref_distance):
    xy = np.column_stack([telemetry["X"].to_numpy(),
                          telemetry["Y"].to_numpy()])
    _, idx = tree.query(xy)
    telemetry = telemetry.copy()
    telemetry["TrackDistance"] = ref_distance[idx]
    return telemetry

# --- Attach telemetry for each overtake ---
overtake_details = []

for _, row in overtakes.iterrows():
    lap = row["LapNumber"]
    passing = row["PassingDriver"]
    passed  = row["PassedDriver"]

    passing_tel = laps[(laps['LapNumber']==lap) & (laps['Driver']==passing)].iloc[0].get_telemetry()
    passed_tel  = laps[(laps['LapNumber']==lap) & (laps['Driver']==passed)].iloc[0].get_telemetry()

    passing_tel = add_track_distance(passing_tel, tree, ref_distance)
    passed_tel  = add_track_distance(passed_tel, tree, ref_distance)

    overtake_details.append({
        "LapNumber": lap,
        "PassingDriver": passing,
        "PassedDriver": passed,
        "PassingTelemetry": passing_tel,
        "PassedTelemetry": passed_tel
    })


core           INFO 	Loading data for British Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['16', '63', '44', '1', '6', '30', '41', '5', '43', '10', '81', '87', '31', '11', '12', '77', '55', '14', '18', '3', '23', '27']


In [25]:
overtake_points = []

for item in overtake_details:
    passing = item["PassingTelemetry"]
    passed  = item["PassedTelemetry"]

    # Align both telemetry frames by time index
    merged = passing[["Time", "TrackDistance", "X", "Y"]].merge(
        passed[["Time", "TrackDistance", "X", "Y"]],
        on="Time",
        suffixes=("_passing", "_passed")
    )

    # Find the first moment passing driver is ahead
    ahead = merged[merged["TrackDistance_passing"] > merged["TrackDistance_passed"]]

    if ahead.empty:
        continue

    overtake_moment = ahead.iloc[0]

    overtake_points.append({
        "LapNumber": item["LapNumber"],
        "PassingDriver": item["PassingDriver"],
        "PassedDriver": item["PassedDriver"],
        "X": overtake_moment["X_passing"],
        "Y": overtake_moment["Y_passing"],
        "TrackDistance": overtake_moment["TrackDistance_passing"]
    })
overtake_points = pd.DataFrame(overtake_points)


In [ ]:
track_x = ref_tel["X"].to_numpy()
track_y = ref_tel["Y"].to_numpy()
track_dist = ref_tel["TrackDistance"].to_numpy()

import numpy as np
import pandas as pd

# Choose bin size (smaller = more detailed)
bin_size = 100  # metres

max_dist = track_dist.max()
bins = np.arange(0, max_dist + bin_size, bin_size)

# Count overtakes per bin
overtake_points["Bin"] = np.digitize(overtake_points["TrackDistance"], bins)

bin_counts = overtake_points.groupby("Bin").size().reindex(range(len(bins)), fill_value=0)

track_bins = np.digitize(track_dist, bins)
segments = []

for i in range(1, len(bins)):
    mask = track_bins == i
    if not mask.any():
        continue

    segments.append({
        "bin": i,
        "x": track_x[mask],
        "y": track_y[mask],
        "count": bin_counts.iloc[i]
    })

In [30]:
import plotly.colors as pc
import plotly.graph_objects as go
import plotly.colors as pc

colorscale = pc.sequential.Reds
def map_to_color(value, vmin, vmax, colorscale):
    # Normalize 0–1
    t = (value - vmin) / (vmax - vmin) if vmax > vmin else 0
    # Pick index
    idx = int(t * (len(colorscale) - 1))
    return colorscale[idx]

vmin = bin_counts.min()
vmax = bin_counts.max()

fig = go.Figure()

for seg in segments:
    color_hex = map_to_color(seg["count"], vmin, vmax, colorscale)

    fig.add_trace(
        go.Scatter(
            x=seg["x"],
            y=seg["y"],
            mode="lines",
            line=dict(color=color_hex, width=6),
            hoverinfo="text",
            text=f"Overtakes: {seg['count']}"
        )
    )

fig.update_layout(
    title="Overtake Density Track Map",
    xaxis=dict(scaleanchor="y", scaleratio=1),
    width=900,
    height=700,
    showlegend=False
)

fig.show()


In [11]:
import pandas as pd
import plotly.graph_objects as go

sector_cols = [
    "Sector1SessionTime",
    "Sector2SessionTime",
    "Sector3SessionTime",
]

# Find the fastest sector time on each lap
leader_sector_times = laps.groupby("LapNumber")[sector_cols].transform("min")

# Calculate gap to fastest sector time
gaps = laps[sector_cols].sub(leader_sector_times)

gaps = gaps.apply(lambda column: column.dt.total_seconds())
gaps.columns = ["GapS1", "GapS2", "GapS3"]

# Create long dataframe
raceplot_long = pd.concat(
    [
        laps[["Driver", "LapNumber"]],
        gaps,
    ],
    axis=1,
).melt(
    id_vars=["Driver", "LapNumber"],
    value_vars=["GapS1", "GapS2", "GapS3"],
    var_name="Sector",
    value_name="GapToLeader",
)

# Give each sector its chronological position
sector_order = {
    "GapS1": 1,
    "GapS2": 2,
    "GapS3": 3,
}

raceplot_long["SectorOrder"] = raceplot_long["Sector"].map(sector_order)

# Position along the continuous race timeline
raceplot_long["RacePosition"] = (
    (raceplot_long["LapNumber"] - 1) * 3
    + raceplot_long["SectorOrder"]
)

# Sort so Plotly connects the points in the correct order
raceplot_long = raceplot_long.sort_values(
    ["Driver", "RacePosition"]
)

driver_order = (
    laps.sort_values("LapNumber")
    .groupby("Driver")["Position"]
    .last()
    .sort_values()
    .index
    .tolist()
)

fig = go.Figure()

for driver in driver_order:

    driver_team = laps.loc[
        laps["Driver"] == driver,
        "Team"
    ].unique()[0]

    df = raceplot_long[
        raceplot_long["Driver"] == driver
    ]

    fig.add_trace(
        go.Scatter(
            x=df["RacePosition"],
            y=df["GapToLeader"],
            mode="lines",
            name=driver,

            line=dict(
                color=f"rgba({config['teams'][driver_team]['drivers'][driver]['colour']})"
            ),

            customdata=df[["LapNumber", "Sector"]],

            hovertemplate=(
                "Lap %{customdata[0]}<br>"
                "%{customdata[1]}<br>"
                "Gap: %{y:.3f}s"
                "<extra>%{fullData.name}</extra>"
            ),
        )
    )

fig.update_layout(
    title="Gap to Leader by Sector",
    xaxis_title="Lap",
    yaxis_title="Gap to Leader (s)",
    height=700,
    width=1000,
)
fig.update_yaxes(autorange="reversed")

fig.show()